In [1]:
!pip install netcdf4 h5netcdf geodatasets xarray


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 77.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [geodatasets] [xarray]]


In [ ]:
import xarray as xr

ds = xr.open_dataset("historical_data.nc", engine="netcdf4")
print(ds)
print(ds.variables.keys())


In [ ]:
df = ds["fwi-nods-gt-30"].to_dataframe().reset_index()
display(df.head())

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from geodatasets import get_path

# Créer la géométrie à partir de lon/lat
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

# Filtrer sur une seule date
gdf_2005 = gdf[gdf["time"] == "2005-11-01"]

# Carte
fig, ax = plt.subplots(figsize=(12, 8))

world = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
world.boundary.plot(ax=ax, linewidth=0.5, color="black")

gdf_2005.plot(
    column="fwi-nods-gt-30",
    ax=ax,
    cmap="YlOrRd",
    legend=True,
    markersize=1,
    legend_kwds={"label": "Nombre de jours FWI > 30"}
)

# Frontières des pays avec geodatasets
world = gpd.read_file(get_path("naturalearth.land"))
world.boundary.plot(ax=ax, linewidth=0.5, color="black")

ax.set_title("FWI - Nombre de jours > 30 (Novembre 2005)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr
import pandas as pd
import s3fs
import glob
import os

data_dir = "data_fwi"

s3_path = "s3://mon-bucket/fwi/output.parquet"

fs = s3fs.S3FileSystem()

dfs = []

for file in glob.glob(os.path.join(data_dir, "*.nc")):
    print(f"Lecture de {file}")
    
    ds = xr.open_dataset(file, engine="netcdf4")
    
    df = ds["VARIABLE"].to_dataframe().reset_index()
    
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

final_df.to_parquet(
    s3_path,
    engine="pyarrow",
    storage_options={"anon": False}
)

print("✅ Données sauvegardées sur S3")

---

In [4]:
import os
import xarray as xr
import s3fs
import pandas as pd

# Chemin vers data_fwi depuis votre notebook
folder_path = '/home/onyxia/work/gen-ai-fwi/data_fwi'
nc_files = [f for f in os.listdir(folder_path) if f.endswith('.nc')]
s3_path = "s3://matheo/diffusion/data/data_1984_1994.parquet"

fs = s3fs.S3FileSystem()
dfs = []

print(f"Nombre de fichiers .nc trouvés : {len(nc_files)}\n")

for nc_file in nc_files:
    file_path = os.path.join(folder_path, nc_file)
    
    try: 
        ds = xr.open_dataset(file_path)
        df = ds["fwi-daily-proj"].to_dataframe().reset_index()
        dfs.append(df)
        print(f"Lecture de : {nc_file}")
    except:
        continue

final_df = pd.concat(dfs, ignore_index=True)

final_df.to_parquet(
    s3_path,
    engine="pyarrow",
    #storage_options={"anon": False}
)

print("✅ Données sauvegardées sur S3")

Nombre de fichiers .nc trouvés : 11

Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19890101_19891231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19940101_19941231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19930101_19931231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19870101_19871231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19900101_19901231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19920101_19921231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19860101_19861231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19910101_19911231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19850101_19851231_v2.nc
Lecture de : eur11_rca4_CNRM-CERFACS-CNRM-CM5_historical_fwi-daily-proj_19880101_19881231_v2.nc
Lec